In [1]:
import os
import pandas as pd
from datasets import load_dataset
from tqdm import tqdm
from dotenv import load_dotenv

In [ ]:
os.environ['HF_TOKEN'] = ':)'

In [3]:
# Disable symlinks warning in datasets
import warnings
warnings.filterwarnings('ignore')

load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")

In [4]:
DS_SIZE = 256 
IMAGES_DIR = "../data/images"
METADATA_PATH = "../data/metadata.csv"

os.makedirs(IMAGES_DIR, exist_ok=True)

In [5]:
def process_stream(stream, target_count, split_name, current_idx, metadata_list):
    glasses_count = 0
    no_glasses_count = 0
    
    print(f"Collecting {target_count} with glasses and {target_count} without for {split_name}...")
    for item in tqdm(stream):
        has_glasses = item['Eyeglasses']
        
        keep = False
        if has_glasses is True and glasses_count < target_count:
            glasses_count += 1
            keep = True
            
        elif has_glasses is False and no_glasses_count < target_count:
            no_glasses_count += 1
            keep = True
            
        if keep:
            img = item['image'].convert("RGB")
            # Resize do standardu ViT (224x224)
            img = img.resize((224, 224))
            filename = f"img_{current_idx:06d}.jpg"
            img_path = os.path.join(IMAGES_DIR, filename)
            img.save(img_path, quality=90)
            
            metadata_list.append({
                'filename': filename,
                'glasses': int(has_glasses), # Zmieniona nazwa klucza w metadanych
                'split': split_name
            })
            current_idx += 1
            
        if glasses_count >= target_count and no_glasses_count >= target_count:
            break
            
    return current_idx

In [6]:
if os.path.exists(METADATA_PATH):
    print("Metadata already exists. Please delete if you want to re-download.")
else:
    print("Streaming data from HuggingFace...")
    ds = load_dataset("flwrlabs/celeba", split="train", streaming=True, token=HF_TOKEN)
    ds_iterator = iter(ds)
    
    metadata_list = []
    idx = 0
    
    # Collect train (set A for CAV)
    idx = process_stream(ds_iterator, 5 * DS_SIZE, "train", idx, metadata_list)
    
    # Collect test (set B for Recovery Analysis)
    idx = process_stream(ds_iterator, 3 * DS_SIZE, "test", idx, metadata_list)
    
    # Save metadata
    df = pd.DataFrame(metadata_list)
    df.to_csv(METADATA_PATH, index=False)
    print(f"Saved metadata with {len(df)} records.")

Streaming data from HuggingFace...


Resolving data files:   0%|          | 0/19 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/19 [00:00<?, ?it/s]

16749it [00:48, 342.64it/s]


2085it [00:07, 295.92it/s]

Saved metadata with 2048 records.
